# Can we predict 12-month loan default?
Target: `defaulted` (1 = defaulted within 12 months of funding). 8,000 applications, ~10% positive.

Plan: inspect the data → baseline → logistic regression vs. gradient boosting with cross-validation → a single held-out test evaluation → interpret. Because the classes are imbalanced, accuracy is meaningless (predicting "no default" gets ~90%), so I use ROC AUC and PR AUC, and report lift over the base rate.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv'); print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data quality

In [ ]:
print(df.shape, '| missing:', df.isna().sum().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
print('default rate: %.3f' % df.defaulted.mean())
df.describe().T

`application_id` is an identifier, so it is dropped from the features (its correlation with the target is ~0.005). There is no date column, so I use a random stratified split rather than a temporal one — see caveats at the end.

## 2. Exploration

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(14,7))
for a,c in zip(ax.ravel(),num):
    q=pd.qcut(df[c],10,duplicates='drop')
    r=df.groupby(q,observed=True).defaulted.mean()
    a.plot(range(len(r)),r.values,marker='o'); a.axhline(df.defaulted.mean(),ls='--',c='grey')
    a.set_title(c); a.set_xlabel('decile'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean','size']))

Default rate by decile of each feature (dashed = overall rate). Monotone trends suggest signal; flat lines suggest noise.

In [ ]:
print(df[num+['defaulted']].corr()['defaulted'].drop('defaulted').sort_values())
print(df[num].corr().round(2))

## 3. Split and models
Hold out 20% as a test set, touched once at the end. Model comparison uses 5-fold stratified CV on the remaining 80%.

In [ ]:
X=df.drop(columns=['defaulted','application_id']); y=df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),['region'])])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic':make_pipeline(pre,LogisticRegression(max_iter=1000)),
 'hist gradient boosting':make_pipeline(ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),['region'])]),
     HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=SEED)),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows={}
for n,m in models.items():
    s=cross_validate(m,Xtr,ytr,cv=cv,scoring=['roc_auc','average_precision'])
    rows[n]={'ROC AUC':s['test_roc_auc'].mean(),'AUC sd':s['test_roc_auc'].std(),'PR AUC':s['test_average_precision'].mean()}
pd.DataFrame(rows).T.round(3)

## 4. Held-out test evaluation

In [ ]:
res={}
for n in ['logistic','hist gradient boosting']:
    models[n].fit(Xtr,ytr); res[n]=models[n].predict_proba(Xte)[:,1]
base=yte.mean()
for n,p in res.items():
    rng=np.random.default_rng(SEED); b=[]
    for _ in range(500):
        i=rng.integers(0,len(yte),len(yte)); b.append(roc_auc_score(yte.values[i],p[i]))
    print(f'{n:24s} AUC {roc_auc_score(yte,p):.3f} (95% CI {np.percentile(b,2.5):.3f}-{np.percentile(b,97.5):.3f})  PR AUC {average_precision_score(yte,p):.3f} (base rate {base:.3f})  Brier {brier_score_loss(yte,p):.4f}')
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
for n,p in res.items():
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
    fp,mp=calibration_curve(yte,p,n_bins=8,strategy='quantile'); ax[2].plot(mp,fp,marker='o',label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR')
ax[1].axhline(base,c='k',ls=':'); ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision')
ax[2].plot([0,.5],[0,.5],'k:'); ax[2].set(title='Calibration',xlabel='mean predicted',ylabel='observed rate')
ax[0].legend(); plt.tight_layout(); plt.show()

### Practical view: what if we focus on the riskiest applicants?

In [ ]:
p=res['logistic']; d=pd.DataFrame({'y':yte.values,'p':p})
d['decile']=pd.qcut(d.p.rank(method='first'),10,labels=False)+1
t=d.groupby('decile').agg(n=('y','size'),default_rate=('y','mean'),avg_pred=('p','mean')).round(3)
t['lift']=(t.default_rate/base).round(2); display(t)
d=d.sort_values('p',ascending=False)
for k in (0.1,0.2,0.3):
    top=d.head(int(k*len(d))); print(f'top {k:.0%} riskiest: default rate {top.y.mean():.1%} vs {base:.1%} overall, captures {top.y.sum()/d.y.sum():.0%} of defaults')

## 5. What drives the prediction?

In [ ]:
m=models['logistic']; m.fit(Xtr,ytr)
coef=pd.Series(m[-1].coef_[0],index=m[0].get_feature_names_out()).sort_values()
print('Logistic coefficients (log-odds per 1 SD for numerics):'); print(coef.round(3))
gb=models['hist gradient boosting'].fit(Xtr,ytr)
pi=permutation_importance(gb,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
imp=pd.Series(pi.importances_mean,index=X.columns); sd=pd.Series(pi.importances_std,index=X.columns)
o=imp.sort_values().index; imp[o].plot.barh(xerr=sd[o]); plt.title('Permutation importance (drop in test AUC), boosting'); plt.show()

## 6. Conclusions
*(Figures below are from running this notebook with `SEED=0`; re-running reproduces them.)*

**Yes, default is predictable, though only moderately.**
- Held-out ROC AUC ≈ **0.78** for logistic regression (95% bootstrap CI ≈ 0.75–0.82) vs. 0.50 for the baseline; PR AUC ≈ 0.31 vs. a 0.10 base rate. 5-fold CV agrees (≈ 0.775 ± 0.014).
- Ranking is useful: the riskiest 10% of applicants default at ~35% (3.4× the base rate) and contain ~34% of all defaults; the riskiest 30% contain ~65%. The safest 20% default at <1%.
- Predicted probabilities are well calibrated, so they can be used as risk estimates and not just for ranking.

**Simple beats complex here.** Gradient boosting did not beat logistic regression (CV AUC 0.769 vs 0.775; test 0.762 vs 0.781, within noise). That points to mostly additive, near-linear effects, so the interpretable model is the one to use.

**Drivers** (logistic, per 1 SD): higher `debt_ratio` (+0.84) and `loan_amount_k` (+0.32) raise risk; higher `income_k` (−0.64) and `employment_years` (−0.36) lower it. `age`, `credit_lines` and `region` carry little or no signal. The features are essentially uncorrelated with each other, so these effects are cleanly separable.

**Caveats**
- Moderate AUC means plenty of individual defaults are not flagged; this suits prioritising/pricing/review, not auto-decisions on single applicants.
- No date column, so the split is random. If the data span time, performance on future applications may be lower (drift); validate out-of-time before deploying.
- Only funded loans are observed: applicants who were rejected have no outcome, so the model reflects the historical approval policy (selection bias).
- Using `region`/`age` in actual credit decisions may be restricted by fair-lending rules; they add little here, so dropping them costs almost nothing.
- Threshold choice (who to flag) depends on the cost of a missed default vs. a declined good loan, which isn't known from this data.
- Not tuned extensively; with ~830 positives, further tuning is unlikely to move AUC much.